# Initial Mean Anomaly and Semi-major Axis Gradient Descent and Gauss-Newton Matrix

## Imports

In [11]:
import importlib
import tobeyLibrary as tobey
importlib.reload(tobey)

import numpy as np
import pandas as pd
from sgp4.api import Satrec, jday
from astropy.time import Time

EARTH_RADIUS = 6378.137 #km

# Binar Groundstation Geodetic Coords
lat_deg = -32.007
lon_deg = 115.894
alt_m = 50

## True TLE to COE

1 60956U 98067WW  24256.99968611  .00235683  00000+0  34489-2 0  9990 \
2 60956  51.6339 240.0861 0011344 359.3789  15.7195 15.54180488  2268

In [12]:
line1 = "1 60956U 98067WW  24256.99968611  .00235683  00000+0  34489-2 0  9990"
line2 = "2 60956  51.6339 240.0861 0011344 359.3789  15.7195 15.54180488  2268"
#                   i       node     e       arg        M       n

satellite = Satrec.twoline2rv(line1, line2)

a_true = satellite.a * EARTH_RADIUS * 1000   # semi-major axis (m)
e_true = satellite.ecco   # eccentricity
i_true = np.rad2deg(satellite.inclo)  # inclination (deg)
node_true = np.rad2deg(satellite.nodeo)  # RAAN (deg)
arg_true = np.rad2deg(satellite.argpo)  # argument of perigee (deg)
M0_true = np.rad2deg(satellite.mo)     # mean anomaly (deg)

jd_total = satellite.jdsatepoch + satellite.jdsatepochF
t0 = Time(jd_total, format='jd', scale='utc')

print("TRUE Classical Orbit Elements")
print("Semi-major axis - a:  ", a_true, "m")
print("Eccentricity - e:     ", e_true)
print("Inclination - i:      ", i_true, "deg")
print("RAAN - node:          ", node_true, "deg")
print("Arg of perigee - arg: ", arg_true, "deg")
print("Mean anomaly - M0:    ", M0_true, "deg")
print("ISO Datetime stamp:   ", t0.isot)

TRUE Classical Orbit Elements
Semi-major axis - a:   6783182.731170822 m
Eccentricity - e:      0.0011344
Inclination - i:       51.6339 deg
RAAN - node:           240.0861 deg
Arg of perigee - arg:  359.3789 deg
Mean anomaly - M0:     15.7195 deg
ISO Datetime stamp:    2024-09-12T23:59:32.880


# Importing Fake Data

In [13]:
doppler_df = pd.read_excel("FakeDopplerMeasurements.xlsx")

# Single Doppler Measurement

In [14]:
# Get first measurement as single data point
doppler_1 = doppler_df.iloc[0].copy(deep=True)

t_obs_1 = tobey.getTime(doppler_1)
elapsed_sec = (t_obs_1 - t0).sec

# Get groundstation position and velocity vectors at this time
r_gs, v_gs = tobey.groundstationECI(lat_deg, lon_deg, alt_m, t_obs_1)

print("Groundstation State Vectors")
print("r_gs:", r_gs)
print("v_gs:", v_gs)

# Orbital Propagation (M0 -> M(t) -> E -> nu)
n_true = tobey.nCalc(a_true)
E_1 = tobey.propagateM02E(elapsed_sec, M0_true, n_true, e_true)
nu_1 = tobey.nuCalc(E_1, e_true)

coe = [a_true, e_true, i_true, node_true, arg_true, nu_1]
print("\nClassical Orbital Elements (a, e, i, RAAN, argp, nu) at t1")
print([f"{v:.6f}" for v in coe])

#COE2RV
r, v, rot_mat = tobey.COE2RV(coe)
print("\nSatellite State Vectors")
print("r:", r)
print("v:", v)

# Vectors and other values needed
rho = tobey.rhoCalc(r, r_gs)
rho_hat = tobey.rho_hatCalc(rho)
v_rel = tobey.v_relCalc(v, v_gs)

k = tobey.kCalc(doppler_1["Centre Frequency (Hz)"])

# Gradients
dfD_dX = tobey.dfD_dXCalc(k, v_rel, rho, rho_hat)
dX_dt = tobey.dX_dtCalc(v, r)

dfD_dM0 = tobey.dfD_dM0Calc(dfD_dX, dX_dt, n_true)
dfD_da = tobey.dfD_daCalc(dfD_dX, dX_dt, elapsed_sec, rot_mat, n_true, a_true, e_true, E_1, nu_1)

# Jacobian
J = tobey.Jacobian(dfD_dM0, dfD_da)
print("\nJacobian:")
print(J)

# Gauss-Newton
G = tobey.GaussNewton(J)
print("\nGauss-Newton Matrix:")
print(G)

# Eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(G)
print("\nEigenvalues:", eigenvalues)
print("Eigenvectors:\n", eigenvectors)

Groundstation State Vectors
r_gs: [-4022202.60251025 -3632551.277273   -3351356.54150797]
v_gs: [ 264.87946656 -292.71712886   -0.62329967]

Classical Orbital Elements (a, e, i, RAAN, argp, nu) at t1
['6783182.731171', '0.001134', '51.633900', '240.086100', '359.378900', '-0.253021']

Satellite State Vectors
r: [-4212828.02458554 -5122773.27931198 -1385576.67790671]
v: [ 2988.95660821 -4026.38896684  5809.18229423]

Jacobian:
[[-3.90148794e+01  1.73738229e-02]]

Gauss-Newton Matrix:
[[ 1.52216082e+03 -6.77837604e-01]
 [-6.77837604e-01  3.01849721e-04]]

Eigenvalues: [   0.         1522.16111866]
Eigenvectors:
 [[-4.45312696e-04 -9.99999901e-01]
 [-9.99999901e-01  4.45312696e-04]]


# Multiple Doppler Measurements

In [15]:
# Build Time array for all observations
t_obs_series = doppler_df.apply(tobey.getTime, axis=1)
t_obs = Time(t_obs_series.tolist())
elapsed_sec_all = (t_obs - t0).sec

n_true = tobey.nCalc(a_true)

# Storage for per-observation results
dfD_dM0_list = []
dfD_da_list = []

for idx in range(len(doppler_df)):
    t_i = t_obs[idx]
    elapsed_sec_i = elapsed_sec_all[idx]

    # Groundstation state at this observation
    r_gs_i, v_gs_i = tobey.groundstationECI(lat_deg, lon_deg, alt_m, t_i)

    # Orbital propagation (M0 -> M(t) -> E -> nu) at this observation
    E_i = tobey.propagateM02E(elapsed_sec_i, M0_true, n_true, e_true)
    nu_i = tobey.nuCalc(E_i, e_true)

    coe_i = [a_true, e_true, i_true, node_true, arg_true, nu_i]
    r_i, v_i, rot_mat_i = tobey.COE2RV(coe_i)

    # Vectors needed for this observation
    rho_i = tobey.rhoCalc(r_i, r_gs_i)
    rho_hat_i = tobey.rho_hatCalc(rho_i)
    v_rel_i = tobey.v_relCalc(v_i, v_gs_i)

    k_i = tobey.kCalc(doppler_df["Centre Frequency (Hz)"].iloc[idx])

    # Gradients at this observation
    dfD_dX_i = tobey.dfD_dXCalc(k_i, v_rel_i, rho_i, rho_hat_i)
    dX_dt_i = tobey.dX_dtCalc(v_i, r_i)

    dfD_dM0_i = tobey.dfD_dM0Calc(dfD_dX_i, dX_dt_i, n_true)
    dfD_da_i = tobey.dfD_daCalc(dfD_dX_i, dX_dt_i, elapsed_sec_i, rot_mat_i, n_true, a_true, e_true, E_i, nu_i)

    dfD_dM0_list.append(dfD_dM0_i)
    dfD_da_list.append(dfD_da_i)

# Jacobian across all observations
J = tobey.Jacobian(dfD_dM0_list, dfD_da_list)
print("\nJacobian Matrix (multiple observations):")
print(J)

# Gauss-Newton
G = tobey.GaussNewton(J)
print("\nGauss-Newton Matrix (multiple observations):")
print(G)

# Eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(G)
print("\nEigenvalues:", eigenvalues)
print("Eigenvectors:\n", eigenvectors)


Jacobian Matrix (multiple observations):
[[-3.90148794e+01  1.73738229e-02]
 [-3.81635335e+01  1.70366553e-02]
 [-3.39951950e+01  1.53850524e-02]
 [-2.73929932e+01  1.27662383e-02]
 [-2.54756615e+01  1.20049932e-02]
 [-2.16484392e+01  1.04843863e-02]
 [-1.98606881e+01  9.77356711e-03]
 [-1.97152834e+01  9.71573808e-03]
 [-1.88794271e+01  9.38326300e-03]
 [-1.87413013e+01  9.32831363e-03]]

Gauss-Newton Matrix (multiple observations):
[[ 7.49313819e+02 -3.47117931e-01]
 [-3.47117931e-01  1.61079748e-04]]

Eigenvalues: [2.78151802e-07 7.49313980e+02]
Eigenvectors:
 [[-4.63247688e-04 -9.99999893e-01]
 [-9.99999893e-01  4.63247688e-04]]
